# 03 — ALT classification (frozen labels) and cohort construction

Builds the three Stage 1 analysis sets by joining the **frozen** ALT labels from
`references/master_manifest.csv` onto the CRISPR-screened osteosarcoma lines.

Per prespec §3, labels are **taken, not re-derived**: `alt_call_direction` gives the arm,
`alt_status` the tier. The 19-gene signature was the *product* of Aim 2, not a classifier
for unlabelled lines.

Every pre-analysis assertion runs here and aborts on failure:
- ground-truth anchors ACH-000364 (U2OS) = positive, ACH-000748 (SJSA1) = negative
- set sizes 16 / 9 / 1
- ALT tier x direction breakdown

Reads no gene-effect values.

## 1 — Header

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T00:27:23Z
Python: 3.12.13


## 2 — Gate check

In [2]:
from src.cohort import (
    build_cohort, assert_all, gold_only, tier_counts,
    EXPECTED_SET_SIZES, EXPECTED_TIER_COUNTS, GROUND_TRUTH_DIRECTION,
)
leaked = [m for m in sys.modules if 'dependency_stats' in m]
assert not leaked, f'OUTCOME LEAK: {leaked} imported in this notebook'
print('Gate check passed: src.dependency_stats is NOT loaded.')
print('CRISPRGeneEffect.csv is unreachable from this notebook.')

Gate check passed: src.dependency_stats is NOT loaded.
CRISPRGeneEffect.csv is unreachable from this notebook.


## 3 — Inclusion filter

In [3]:
cohort = build_cohort()
print('\n'.join(cohort.trace.as_lines()))
print()
print(f'sizes: {cohort.sizes}')

  Model.csv rows matching 'steosarcoma'                          N = 35
  present in CRISPRGeneEffect.csv (screened)                     N = 26
  present in expression matrix                                   N = 26
  joined to frozen manifest (cohort == 'DepMap_26Q1')            N = 26
  alt_status in ('alt_gold_positive', 'alt_silver', 'alt_negative_high_conf') N = 25
  alt_call_direction in ('positive', 'negative')                 N = 25
  ALT+ (primary contrast set)                                    N = 16
  ALT- (negative-control set)                                    N = 9

sizes: {'alt_positive': 16, 'alt_negative': 9, 'excluded_unknown': 1}


## 4 — Pre-analysis assertions

In [4]:
assert_all(cohort)   # anchors, set sizes, tier breakdown -- aborts on any mismatch
print('All pre-analysis assertions PASSED.\n')
for model_id, expected in GROUND_TRUTH_DIRECTION.items():
    row = cohort.all_screened().set_index('ModelID').loc[model_id]
    print(f'  anchor {model_id} ({row["StrippedCellLineName"]:>8s}) '
          f'-> {row["alt_call_direction"]}  [expected {expected}]')
print()
print('tier x direction:')
for (status, direction), n in sorted(tier_counts(cohort).items()):
    print(f'  {status:<24s} {direction:<9s} {n}')

All pre-analysis assertions PASSED.

  anchor ACH-000364 (    U2OS) -> positive  [expected positive]
  anchor ACH-000748 (   SJSA1) -> negative  [expected negative]

tier x direction:
  alt_gold_positive        positive  10
  alt_negative_high_conf   negative  7
  alt_silver               negative  2
  alt_silver               positive  6


## 5 — The three analysis sets

In [5]:
COLS = ['ModelID', 'StrippedCellLineName', 'alt_status', 'alt_source',
        'atrx_status', 'telomerase_status', 'tp53_status']
print(f'=== ALT+ primary contrast set (N={len(cohort.alt_positive)}) ===')
print(cohort.alt_positive[COLS].to_string(index=False))
print(f'\n=== ALT- negative control (N={len(cohort.alt_negative)}) ===')
print(cohort.alt_negative[COLS].to_string(index=False))
print(f'\n=== Excluded (N={len(cohort.excluded_unknown)}) ===')
print(cohort.excluded_unknown[['ModelID', 'StrippedCellLineName', 'alt_status']].to_string(index=False))

gold = gold_only(cohort.alt_positive)
print(f'\nR1 gold-only ALT+ sensitivity cohort: N={len(gold)}')

=== ALT+ primary contrast set (N=16) ===
   ModelID StrippedCellLineName        alt_status              alt_source atrx_status telomerase_status tp53_status
ACH-000082      G292CLONEA141B1 alt_gold_positive           ALTitude_2024 not_altered     indeterminate not_altered
ACH-000364                 U2OS alt_gold_positive           ALTitude_2024 not_altered     indeterminate not_altered
ACH-000410                SAOS2 alt_gold_positive           ALTitude_2024 not_altered     indeterminate not_altered
ACH-001526                 HUO9 alt_gold_positive           ALTitude_2024 not_altered     indeterminate not_altered
ACH-001715                CAL72 alt_gold_positive           ALTitude_2024 not_altered     indeterminate not_altered
ACH-002067                 NOS1 alt_gold_positive           ALTitude_2024 not_altered     indeterminate     altered
ACH-002433        CCLFPEDS0019T        alt_silver ALTitude_2024_predicted not_altered     indeterminate     altered
ACH-002471                OS052

## 6 — R5 feasibility: ATRX status

Prespec §7 records in advance that R5 is expected to be near-vacuous. Confirmed here
rather than discovered during robustness.

In [6]:
atrx = cohort.alt_positive['atrx_status'].value_counts()
print('ATRX status among ALT+ lines:')
print(atrx.to_string())
n_altered = int((cohort.alt_positive['atrx_status'] == 'altered').sum())
print(f'\nATRX-altered: {n_altered} of {len(cohort.alt_positive)}')
print(f'ATRX-wildtype subset for R5: {len(cohort.alt_positive) - n_altered} of {len(cohort.alt_positive)}')
if n_altered <= 1:
    print('\n-> R5 is near-vacuous as prespec §7 anticipated. It will be reported as')
    print('   uninformative, NOT as evidence of no ATRX confounding.')

ATRX status among ALT+ lines:
atrx_status
not_altered    15
altered         1

ATRX-altered: 1 of 16
ATRX-wildtype subset for R5: 15 of 16

-> R5 is near-vacuous as prespec §7 anticipated. It will be reported as
   uninformative, NOT as evidence of no ATRX confounding.


## 7 — Persist and report

In [7]:
from src.utils import PROCESSED_DIR

sets = []
for name, frame in [('alt_positive', cohort.alt_positive),
                    ('alt_negative', cohort.alt_negative),
                    ('excluded_unknown', cohort.excluded_unknown)]:
    tagged = frame.copy()
    tagged.insert(0, 'analysis_set', name)
    sets.append(tagged)
cohort_path = PROCESSED_DIR / 'cohort_alt_sets.csv'
pd.concat(sets, ignore_index=True).to_csv(cohort_path, index=False)

lines = [
    '# Notebook 03 — ALT classification and cohort construction',
    '',
    'Labels taken from `references/master_manifest.csv` (frozen). Never re-derived.',
    'Reads no gene-effect values.',
    '',
    '## Inclusion filter',
    '',
] + cohort.trace.as_markdown() + [
    '',
    '## Analysis sets',
    '',
    f'- ALT+ primary contrast: **N={len(cohort.alt_positive)}** '
    f'({len(gold)} gold + {len(cohort.alt_positive) - len(gold)} silver)',
    f'- ALT- negative control: **N={len(cohort.alt_negative)}**',
    f'- Excluded (`alt_unknown`): **N={len(cohort.excluded_unknown)}**',
    '',
    f'Total {sum(cohort.sizes.values())} = every CRISPR-screened osteosarcoma line.',
    '',
    '## Pre-analysis assertions',
    '',
    'All PASSED: ground-truth anchors, set sizes, tier x direction breakdown.',
    '',
    '| Anchor | Expected | Observed |',
    '|---|---|---|',
]
for model_id, expected in GROUND_TRUTH_DIRECTION.items():
    row = cohort.all_screened().set_index('ModelID').loc[model_id]
    lines.append(f'| {model_id} ({row["StrippedCellLineName"]}) | {expected} | {row["alt_call_direction"]} |')
lines += [
    '',
    '| ALT tier | Direction | N |',
    '|---|---|---|',
]
for (status, direction), n in sorted(tier_counts(cohort).items()):
    lines.append(f'| {status} | {direction} | {n} |')
lines += [
    '',
    '## ALT+ primary contrast set',
    '',
    '| ModelID | Cell line | ALT tier | ATRX | Telomerase |',
    '|---|---|---|---|---|',
]
for _, r in cohort.alt_positive.iterrows():
    lines.append(f"| {r['ModelID']} | {r['StrippedCellLineName']} | {r['alt_status']} "
                 f"| {r['atrx_status']} | {r['telomerase_status']} |")
lines += [
    '',
    '## ALT- negative control set',
    '',
    '| ModelID | Cell line | ALT tier | ATRX | Telomerase |',
    '|---|---|---|---|---|',
]
for _, r in cohort.alt_negative.iterrows():
    lines.append(f"| {r['ModelID']} | {r['StrippedCellLineName']} | {r['alt_status']} "
                 f"| {r['atrx_status']} | {r['telomerase_status']} |")
lines += [
    '',
    '## R5 (ATRX control) feasibility',
    '',
    f'ATRX-altered ALT+ lines: **{n_altered} of {len(cohort.alt_positive)}**. The '
    f'ATRX-wildtype subset is {len(cohort.alt_positive) - n_altered} of '
    f'{len(cohort.alt_positive)}, so R5 cannot meaningfully control for ATRX. Prespec §7 '
    'recorded this expectation in advance; R5 will be reported as uninformative rather '
    'than as evidence of no ATRX confounding.',
    '',
    'Likely cause: Aim 2 derived `atrx_status` from damaging point mutations only',
    '(`OmicsSomaticMutationsMatrixDamaging`), which misses ATRX deletions and loss of',
    'expression -- the dominant ATRX-inactivation routes in ALT.',
]
report = write_report('03', lines)
manifest = write_sha256_manifest('03', [report, cohort_path])
record_provenance([{
    'notebook': '03', 'run_timestamp': RUN_TIMESTAMP, 'sizes': cohort.sizes,
    'gold_only_alt_positive': int(len(gold)), 'atrx_altered_alt_positive': n_altered,
}])
print(f'wrote {cohort_path.name}')
print(f'wrote {report.name}')
print(f'wrote {manifest.name}')

wrote cohort_alt_sets.csv
wrote notebook_03_report.md
wrote notebook_03_sha256.json
